# SynthAug - Colab GPU Backend
Run each cell in order. Keep the last cell running to maintain the server.

In [ ]:
# Cell 1: Install Dependencies & Clone Engine 🛠️
# ⚠️ CRITICAL: We pin timm to 0.4.12 to avoid "AttributeError: act1"
!pip install pyngrok flask flask-cors timm==0.4.12
!git clone https://github.com/autonomousvision/projected_gan.git
print("✅ Dependencies installed & Engine cloned.")

In [ ]:
# Cell 2: Imports & Configuration ⚙️
import os
import shutil
import subprocess
import threading
import time
import zipfile
from flask import Flask, request, jsonify, send_file
from flask_cors import CORS
from pyngrok import ngrok
from werkzeug.utils import secure_filename

# --- 🔑 PASTE YOUR TOKEN BELOW ---
# Go to dashboard.ngrok.com -> Your Authtoken
NGROK_AUTH_TOKEN = "30vKMmxuO5VZ1z8Di1DWNW0rNj7_5GynJFkFSjNKcMZSGa8cp"

# Setup Flask
app = Flask(__name__)
CORS(app) # Allow cross-origin requests

# Global State
TRAINING_STATE = {
    "status": "idle",
    "progress": 0,
    "log": "System Online. Waiting for uplink..."
}

print("✅ Config loaded.")

In [ ]:
# Cell 3: The Auto-Patcher 🩹
def patch_projected_gan():
    """
    Automatically fixes the 'betas=[0, 0.99]' error in train.py
    by converting the integer 0 to float 0.0
    """
    train_script = "/content/projected_gan/train.py"
    if os.path.exists(train_script):
        with open(train_script, 'r') as f:
            code = f.read()

        # Fix both formats: with and without spaces
        code = code.replace("betas=[0, 0.99]", "betas=[0.0, 0.99]")
        code = code.replace("betas=[0,0.99]", "betas=[0.0,0.99]")
        
        with open(train_script, 'w') as f:
            f.write(code)
        print("✅ Patched train.py: Fixed 'betas' integer error.")
    else:
        print("❌ Warning: train.py not found. Did you run Cell 1?")

# Run the patch immediately
patch_projected_gan()

In [ ]:
# Cell 4: Flask API Routes 🔗
import glob

@app.route('/upload', methods=['POST'])
def upload_file():
    if 'file' not in request.files:
        return jsonify({"error": "No file uploaded"}), 400

    file = request.files['file']
    if file.filename == '':
        return jsonify({"error": "No file selected"}), 400

    # Reset workspaces
    paths = ["/content/uploads", "/content/results", "/content/generated"]
    for p in paths:
        if os.path.exists(p):
            shutil.rmtree(p)
        os.makedirs(p)

    # Save and Unzip
    zip_path = os.path.join("/content/uploads", "raw_data.zip")
    file.save(zip_path)

    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall("/content/uploads")
        os.remove(zip_path)

        # Cleanup __MACOSX junk
        macosx = os.path.join("/content/uploads", "__MACOSX")
        if os.path.exists(macosx):
            shutil.rmtree(macosx)

        return jsonify({"message": "Dataset uploaded successfully."}), 200
    except Exception as e:
        return jsonify({"error": str(e)}), 500

@app.route('/train', methods=['POST'])
def start_training():
    data = request.json or {}
    preset = data.get("preset", "draft")
    count = data.get("count", 500)

    if TRAINING_STATE['status'] == 'training':
        return jsonify({"error": "GPU is busy!"}), 409

    thread = threading.Thread(target=run_training_background, args=(preset, count))
    thread.start()

    return jsonify({"message": f"Training started in {preset} mode."}), 200

@app.route('/status', methods=['GET'])
def get_status():
    return jsonify(TRAINING_STATE)

@app.route('/download', methods=['GET'])
def download_results():
    download_type = request.args.get('type', 'generated')

    if download_type == 'merged':
        # Original + Generated combined
        merge_dir = '/content/merged_payload'
        if os.path.exists(merge_dir):
            shutil.rmtree(merge_dir)
        os.makedirs(merge_dir)

        # Copy original images
        for root, dirs, files_list in os.walk('/content/uploads'):
            for file in files_list:
                if file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    src = os.path.join(root, file)
                    shutil.copy2(src, merge_dir)

        # Copy generated images
        if os.path.exists('/content/generated'):
            for file in os.listdir('/content/generated'):
                if file.endswith('.png'):
                    src = os.path.join('/content/generated', file)
                    shutil.copy2(src, merge_dir)

        shutil.make_archive('/content/payload', 'zip', merge_dir)
    else:
        # Generated only
        if not os.path.exists('/content/generated') or not os.listdir('/content/generated'):
            return jsonify({"error": "No generated images found."}), 404
        shutil.make_archive('/content/payload', 'zip', '/content/generated')

    return send_file('/content/payload.zip', as_attachment=True)

def run_training_background(preset, count):
    global TRAINING_STATE
    TRAINING_STATE["status"] = "training"
    TRAINING_STATE["progress"] = 0

    kimg = 50 if preset == "draft" else 600
    snap = 10 if preset == "draft" else 50

    try:
        # PHASE 1: Preprocess
        TRAINING_STATE["log"] = ">> PHASE 1: Preprocessing dataset..."
        if os.path.exists("/content/dataset.zip"):
            os.remove("/content/dataset.zip")

        # Find the actual dataset folder
        upload_contents = os.listdir("/content/uploads")
        dataset_path = "/content/uploads"
        for item in upload_contents:
            item_path = os.path.join("/content/uploads", item)
            if os.path.isdir(item_path):
                dataset_path = item_path
                break

        # Manually resize images to 256x256 (Kaggle method)
        from PIL import Image
        import torchvision.transforms as transforms
        
        raw_dir = '/content/raw'
        if os.path.exists(raw_dir):
            shutil.rmtree(raw_dir)
        os.makedirs(raw_dir)
        
        resize = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(256)])
        cnt = 0
        for root, dirs, files in os.walk(dataset_path):
            for f in files:
                if f.lower().endswith(('.jpg', '.png', '.jpeg')):
                    try:
                        img_path = os.path.join(root, f)
                        img = Image.open(img_path).convert('RGB')
                        resize(img).save(f'{raw_dir}/img{cnt:05d}.jpg')
                        cnt += 1
                    except: pass
        
        if cnt == 0:
            raise Exception('No valid images found!')
        
        subprocess.run(
            ["python", "dataset_tool.py", "--source", raw_dir, "--dest", "/content/dataset.zip"],
            cwd="/content/projected_gan", check=True
        )

        # PHASE 2: Train
        TRAINING_STATE["log"] = f">> PHASE 2: Training ({kimg} kimg)..."
        cmd = [
            "python", "train.py",
            "--outdir=/content/results",
            "--data=/content/dataset.zip",
            "--gpus=1",
            "--cfg=fastgan",
            "--batch=16",
            f"--kimg={kimg}",
            f"--snap={snap}"
        ]

        process = subprocess.Popen(
            cmd, cwd="/content/projected_gan",
            stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
        )

        for line in iter(process.stdout.readline, ''):
            if "tick" in line:
                TRAINING_STATE["log"] = line.strip()
                try:
                    current = int(line.split("tick")[1].split()[0])
                    TRAINING_STATE["progress"] = int((current / kimg) * 90)
                except: pass
        process.wait()

        # PHASE 3: Generate images
        TRAINING_STATE["log"] = ">> PHASE 3: Generating images..."
        TRAINING_STATE["progress"] = 95

        # Find model using Kaggle method
        results_dir = '/content/results'
        run_folders = sorted([f for f in os.listdir(results_dir) if os.path.isdir(os.path.join(results_dir, f))])
        
        if not run_folders:
            raise Exception('No run folder found!')
        
        latest_run = os.path.join(results_dir, run_folders[-1])
        pkl_files = [f for f in os.listdir(latest_run) if f.endswith('.pkl')]
        
        model_pkl = None
        for p in pkl_files:
            if 'network-snapshot' in p:
                model_pkl = os.path.join(latest_run, p)
                break
        
        if not model_pkl and pkl_files:
            model_pkl = os.path.join(latest_run, pkl_files[-1])
        
        if not model_pkl:
            raise Exception('No model file found!')

        gen_cmd = [
            "python", "gen_images.py",
            f"--network={model_pkl}",
            f"--seeds=0-{count}",
            "--outdir=/content/generated"
        ]
        subprocess.run(gen_cmd, cwd="/content/projected_gan", check=True)

        TRAINING_STATE["status"] = "complete"
        TRAINING_STATE["log"] = f">> COMPLETE! Generated {count} images."
        TRAINING_STATE["progress"] = 100

    except Exception as e:
        TRAINING_STATE["status"] = "error"
        TRAINING_STATE["log"] = f"ERROR: {str(e)}"

print("✅ API Routes registered: /upload, /train, /status, /download")

In [ ]:
# Cell 5: START SERVER (Keeps running) 🚀
ngrok.set_auth_token(NGROK_AUTH_TOKEN)
public_url = ngrok.connect(5000).public_url
print(f"🔥 BINDING SUCCESSFUL.")
print(f"🔗 COPY THIS URL INTO YOUR WEBSITE .ENV FILE: {public_url}")

# This blocks the cell so it keeps running
app.run(port=5000)